# Step 5: Work with GeoJSON Regions and Corridors


GeoJSON polygons let you ask targeted spatial questions: which stations are inside a basin, which event-station paths cross a boundary, and how residuals behave inside a corridor. In this notebook, you will load the example regions, make regional residual plots, build corridor selections, and inspect the waveforms behind one boundary-crossing subset.


## Imports

These imports load GeoJSON regions, build corridor selections, and create the maps and record sections used in this notebook.

In [ ]:
from pathlib import Path
import runpy

# Make the local source checkout importable when running notebooks without an installed wheel.
_bootstrap = next(
    (
        path
        for candidate in (Path.cwd(), *Path.cwd().parents)
        for path in (
            candidate / "_source_bootstrap.py",
            candidate / "docs" / "examples" / "_source_bootstrap.py",
        )
        if path.exists()
    ),
    None,
)
if _bootstrap is None:
    raise RuntimeError("Could not find docs/examples/_source_bootstrap.py.")
repo_root = runpy.run_path(str(_bootstrap))["use_source_checkout"]()

from spatial_vtk.config import (
    notebook_timer,
    notebook_figure_settings,
    prepare_notebook_geospatial_environment,
    register_svtk_cell_timer,
)
prepare_notebook_geospatial_environment()

with notebook_timer():
    from IPython.display import display

    from spatial_vtk.spatial import load_standard_geojson_plotting_inputs
    register_svtk_cell_timer()


## Configuration

Load the tutorial run scenario, then set the region and corridor choices used below. These choices are examples; you can swap in your own polygon names, event IDs, and corridor dimensions.

In [ ]:
from spatial_vtk.config import notebook_figure_settings, notebook_run_context

config_path = repo_root / "data/examples/configuration/example_spatial_vtk_config.yaml"

# Load the tutorial run scenario and make it active for Spatial-VTK helper calls.
context = notebook_run_context(config_path, run_scenario="tutorial")
cfg = context.cfg

waveform_figure_settings = notebook_figure_settings("waveform")
spatial_figure_settings = notebook_figure_settings("spatial")
value_column = "log2_residual"
passbands = ["1-2 sec", "2-3 sec"]
component = "Z"

# These regions and anchors are chosen from the example GeoJSON and metadata.
boundary_region = "LA Basin"
station_region = "LA Basin"
event_region = "Glendale"
through_anchor_station = "OLI"
outward_event_id = "ci38695658"
corridor_station_region = boundary_region


## Load GeoJSON Regions and Tutorial Tables

Start by loading the prepared tables from the earlier notebooks and the larger metrics snapshot used for plotting examples.

In [ ]:
# Load compact workflow inputs through one package helper.
geojson_inputs = load_standard_geojson_plotting_inputs(cfg=cfg)

display(geojson_inputs.status_frame())


## Visualize the GeoJSON Regions

Plot the polygons with stations and events on the same basemap. This is the quickest way to check that the GeoJSON file overlaps your project area.

In [ ]:
# Render the GeoJSON overview, regional boxplot, and regional PGA station map.
# The package helper owns GeoJSON annotation, configured figure paths, sidecar kwargs,
# and the region summary table so this notebook stays focused on the spatial workflow.
geojson_region_result = geojson_inputs.write_region_figures(
    settings=spatial_figure_settings,
    value_col=value_column,
    passbands=passbands,
    component=component,
    station_region=station_region,
    event_region=event_region,
)
metrics_by_regions = geojson_region_result.metrics_by_regions

display(geojson_region_result.preview_frame())
display(geojson_region_result.summary_frame())
display(geojson_region_result.status_frame())


## Regional PGA Contrast

Add station-region labels from the GeoJSON file, then compare PGA residuals across stations that fall inside one of the example polygons. The table below the plot compares each mapped region against the LA Basin baseline.


## Residual Map for Events and Stations in Different Regions

Here you can ask a more targeted question: for events in one polygon and stations in another, where are the average station residuals high or low? This example uses Glendale events recorded by LA Basin stations.


## Corridor and Record-Section Figures

The package helper builds the boundary corridors, selects matching event-station paths, loads the QC-passed waveform pairs for the record section, and writes the corridor maps through the configured Step 5 output group.


In [ ]:
# Render corridor maps, the boundary-crossing record section, and the outward-corridor PGV map.
# The helper owns corridor construction, selected-record joins, waveform selection, metric filtering,
# configured figure paths, sidecar kwargs, and compact preview tables.
geojson_corridor_result = geojson_inputs.write_corridor_figures(
    metrics_by_regions=metrics_by_regions,
    spatial_settings=spatial_figure_settings,
    waveform_settings=waveform_figure_settings,
    value_col=value_column,
    passbands=passbands,
    component=component,
    boundary_region=boundary_region,
    through_anchor_station=through_anchor_station,
    outward_event_id=outward_event_id,
    corridor_station_region=corridor_station_region,
    display_func=display,
)

display(geojson_corridor_result.boundary_crossing_frame())
display(geojson_corridor_result.outward_event_frame())
display(geojson_corridor_result.status_frame())
